# DeepSeek-R1 모델 시연


## 시연 목표

- Qwen-1.5B 모델에 DeepSeek-R1 모델을 증류(Distillation)한 DeepSeek-R1-Distill-Qwen-1.5B 모델을 HuggingFace에서 불러와서 사용해봅니다.
- DeepSeek-R1-Distill-Qwen-1.5B 모델이 추론 과정을 출력하는 것을 확인하고, LLM이 사람의 추론 과정을 모방하여 출력할 수 있음을 확인합니다.

In [ ]:
import os
os.environ["HF_HOME"] = "/mnt/elice/dataset"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델과 Tokenizer를 모두 불러옵니다.
# DeepSeek R1 모델 원본은 전체 파라미터 크기가 671B (6710억) 이고, 활성화 파라미터는 37B (370억) 라서 여기서 사용해볼 수는 없습니다.
# 대신, 1.5B 버전의 Distill 모델을 사용합니다.
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda:0")

DeepSeek-R1 논문에서 제시한 DeepSeek-R1-Zero 모델을 위한 프롬프트 템플릿을 그대로 사용하여 입력한 프롬프트에 답변하는 함수를 구현합니다.

이때, 모델이 추론 과정을 출력하도록 강제하기 위해 템플릿 뒤에 `<think>` 텍스트를 추가합니다. 

In [ ]:
def generate_with_cot(prompt):
    # 프롬프트 가장 마지막의 <think> 태그는 논문에 없는 부분입니다.
    cot_prompt = f"A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: {prompt}. Assistant: <think> "

    # 템플릿을 적용한 프롬프트를 토큰화합니다.
    inputs = tokenizer(cot_prompt, return_tensors="pt").to("cuda:0")

    # 답변을 생성합니다.
    outputs = model.generate(**inputs, max_new_tokens=4096, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    assistant_start_marker = "Assistant:"

    if assistant_start_marker in response:
        response = response.split(assistant_start_marker, 1)[1].strip()
    else:
        response = response.strip()

    return response

한번 사용해봅시다.

In [ ]:
# Example Usage
prompt = "What is the capital of France?"
response = generate_with_cot(prompt)
print("Full Response:")
print(response)

In [ ]:
prompt_math = "If a train travels at 60 mph for 3 hours, how far does it travel?"
response_math = generate_with_cot(prompt_math)
print("Full Response:")
print(response_math)

길이 제한으로 인해 답변을 제대로 생성하진 못했지만, `<think>` 태그나 `<answer>` 태그를 통한 추론 과정을 살펴볼 수 있었습니다.

이외에도 원하는 질문을 입력하고, 추론 과정을 살펴봅시다.

In [ ]:
custom_prompt = None
while custom_prompt != "exit":
    custom_prompt = input("프롬프트를 입력하세요 (혹은 'exit' 를 입력해서 종료하세요): ")
    if custom_prompt.lower() == "exit":
        break
    response_custom = generate_with_cot(custom_prompt)
    print("Full Response:")
    print(response_custom)

이번 과정에서는 이처럼 강화학습을 활용하여 추론 기능을 가진 DeepSeek-R1 모델에 대해 살펴보고, 마지막에는 Distillation, 강화학습을 각각 활용하여 작은 크기의 LLM을 학습시키는 실습 까지 진행해 보겠습니다.